In [1]:
import os
import sys
import glob
import chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')

if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion import PyMuPDFReader

load_dotenv()
API_KEY = os.getenv("YANDEX_API_KEY")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")

if not API_KEY or not FOLDER_ID:
    raise ValueError("Ключи не найдены! Проверь файл .env")

print("Загрузка модели эмбеддингов (GPU)...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

print("Инициализация ChromaDB...")
client = chromadb.Client()

try:
    client.delete_collection("materials_db_notebook")
except:
    pass
    
collection = client.create_collection("materials_db_notebook")

print("\nНачинаем парсинг реальных документов...")
raw_data_dir = os.path.join(project_root, "data", "raw")

pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))

if not pdf_files:
    raise ValueError(f"PDF файлы не найдены в папке {raw_data_dir}!")

reader = PyMuPDFReader()

documents = []
metadatas = []
ids = []

for pdf_path in pdf_files:
    print(f" Читаем: {os.path.basename(pdf_path)}")

    chunks = reader.read(pdf_path)
    
    for chunk in chunks:
        documents.append(chunk.text)
        ids.append(chunk.chunk_id)
        
        metadatas.append({
            "source_id": chunk.metadata.source_id,
            "title": chunk.metadata.title,
            "source_type": chunk.metadata.source_type
        })

print(f"\nИзвлечено {len(documents)} фрагментов (страниц). Начинаем векторизацию...")

embeddings = embed_model.encode(documents, batch_size=32, show_progress_bar=True).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("База знаний успешно заполнена реальными статьями и готова к поиску!")

Загрузка модели эмбеддингов (GPU)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Инициализация ChromaDB...

Начинаем парсинг реальных документов...
 Читаем: geokniga-tehnologiyaobogashcheniyapoleznyhiskopaemyh.pdf
 Читаем: geokniga-flotacionnye-metody-obogashcheniya_0.pdf
 Читаем: tehnologiya_izvlecheniya_zolota_i_serebra_iz_upornogo_zolotosoderzhaschego.pdf
 Читаем: geokniga_lodeyshchikovvvtehnologiyaizvlecheniyazolotaiserebraizupornyh1.pdf
 Читаем: geokniga-metallurgiya-blagorodnyh-metallov_0.pdf

Извлечено 609 фрагментов (страниц). Начинаем векторизацию...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

База знаний успешно заполнена реальными статьями и готова к поиску!


In [3]:
import requests

def generate_hypothesis_pipeline(problem, constraints):
    print(f"ЗАПУСК ПАЙПЛАЙНА")
    print(f"Проблема: {problem}")
    print(f"Ограничения: {constraints}\n")
    
    print("Ищем релевантный контекст в базе...")

    query_text = f"{problem}. Ограничения: {constraints}"
    query_vector = embed_model.encode(query_text).tolist()
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=2
    )
    
    retrieved_context = "\n- ".join(results['documents'][0])
    print("Найденный контекст:")
    print(f"- {retrieved_context}\n")
    
    print("Отправка промпта в Yandex AI Studio...")
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    
    system_prompt = (
        "Ты — научный ассистент НИИ. Твоя задача — сгенерировать проверяемую научную гипотезу "
        "строго на основе предоставленного контекста. "
        "Ответ структурируй: 1. Гипотеза 2. Обоснование (с опорой на контекст) 3. Риски 4. Ожидаемый эффект."
    )
    
    user_prompt = f"Проблема: {problem}\nОграничения: {constraints}\n\nКонтекст из базы знаний:\n{retrieved_context}"
    
    payload = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {"stream": False, "temperature": 0.2, "maxTokens": 2000},
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": user_prompt}
        ]
    }
    
    headers = {"Content-Type": "application/json", "Authorization": f"Api-Key {API_KEY}"}
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        answer = response.json()['result']['alternatives'][0]['message']['text']
        print("Ответ модели:\n")
        print("="*50)
        print(answer)
        print("="*50)
    else:
        print(f"Ошибка API {response.status_code}: {response.text}")

target_problem = "Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов"
user_constraints = "Бюджет ограничен, необходимо избегать дорогих компонентов"

# Запуск
generate_hypothesis_pipeline(target_problem, user_constraints)

ЗАПУСК ПАЙПЛАЙНА
Проблема: Уменьшения вредного влияния на процесс цианирования руды сурьмянистых и мышьяковистых минералов
Ограничения: Бюджет ограничен, необходимо избегать дорогих компонентов

Ищем релевантный контекст в базе...
Найденный контекст:
- - применение обычного процесса цианирования станет нерентабель ным . Вместе с увеличением расхода цианида присутствие в рабочих растворах комплексных анионов меди сопровождается заметным уменьшением скорости растворения золота . Для извлечения золота из медистых руд прибегают к специальным методам переработки . 

Характерно , что процесс взаимодействия минералов меди с циа - - нистыми растворами резко замедляется при уменьшении концентра ции цианида . Это иногда используют для извлечения золота из меди - стых руд , обрабатывая их слабыми цианистыми растворами , – метод обратного выщелачивания . 

#### Взаимодействие соединений мышьяка и сурьмы с цианистыми растворами 

Минералы мышьяка и сурьмы часто встречаются в золотых рудах , некотор